# Semana 2: Tensores - La Estrella del Data Science
## Notebook de Ejercicios (NB2) - Dataset CIFAR-10

### Propósito de la sesión
Aplicar los conceptos de tensores, operaciones de manipulación y normas a un dataset real de imágenes. Trabajaremos con CIFAR-10, un conjunto de 60,000 imágenes a color de 32x32 píxeles en 10 clases, para entender cómo se organiza la información visual en estructuras tensoriales y qué implicaciones tiene su manipulación.

### Objetivos de aprendizaje
*   Cargar y explorar un dataset de imágenes real (CIFAR-10) y verificar su estructura como tensor 4D.
*   Visualizar imágenes y sus canales de color (RGB) como tensores 3D.
*   Realizar operaciones de slicing y manipulación de canales.
*   Aplanar (flatten) imágenes y comprender la pérdida de estructura espacial.
*   Reflexionar sobre la importancia de la estructura tensorial en el diseño de arquitecturas de deep learning.

## Configuración Inicial

Importamos las librerías necesarias y cargamos CIFAR-10 directamente desde `keras` o `torchvision`. Usaremos Keras por su simplicidad.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Para cargar CIFAR-10
try:
    from tensorflow.keras.datasets import cifar10
    print("✅ Keras importado correctamente.")
except ImportError:
    print("❌ TensorFlow/Keras no encontrado. Instalando...")
    !pip install tensorflow
    from tensorflow.keras.datasets import cifar10
    print("✅ TensorFlow/Keras instalado e importado.")

# Configuración de estilo para gráficos
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("\n🎯 Librerías listas. Cargando dataset CIFAR-10...")

---
## Actividad 1: Cargar el dataset y verificar su forma como tensor 4D

CIFAR-10 contiene 60,000 imágenes a color de 32x32 píxeles, divididas en 50,000 para entrenamiento y 10,000 para prueba.

La forma de los datos es: **(num_muestras, alto, ancho, canales)**.

Esto es un **tensor de orden 4**:
*   Dimensión 0: Lote (batch) de imágenes
*   Dimensión 1: Alto (32 píxeles)
*   Dimensión 2: Ancho (32 píxeles)
*   Dimensión 3: Canales (3: RGB)

Verifiquemos esta estructura.

In [ ]:
# Cargamos el dataset
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Exploramos las dimensiones
print("🔷 CIFAR-10 - Dataset de imágenes a color")
print(f"  Tensor de entrenamiento (X_train) - forma: {X_train.shape}")
print(f"  Tensor de prueba (X_test) - forma: {X_test.shape}")
print(f"  Etiquetas de entrenamiento (y_train) - forma: {y_train.shape}")

# Verificamos los tipos de datos y rangos
print(f"\n  Tipo de dato de X_train: {X_train.dtype}")
print(f"  Rango de valores en X_train: [{X_train.min()}, {X_train.max()}]")

# Nombres de las clases en CIFAR-10
class_names = ['avión', 'automóvil', 'pájaro', 'gato', 'ciervo',
               'perro', 'rana', 'caballo', 'barco', 'camión']
print(f"\n  Clases: {class_names}")

# Mostramos información sobre el tensor
print(f"\n📊 Estadísticas del tensor 4D:")
print(f"  Número de dimensiones (ndim): {X_train.ndim}")
print(f"  Tamaño total de elementos: {X_train.size:,}")
print(f"  Memoria aproximada: {X_train.nbytes / 1e6:.2f} MB")

### Visualización de muestras del tensor 4D

Visualicemos algunas imágenes del tensor para entender qué representan estos números.

In [ ]:
# Función para mostrar varias imágenes
def mostrar_imagenes(imagenes, etiquetas, class_names, num_imagenes=10):
    plt.figure(figsize=(15, 6))
    for i in range(num_imagenes):
        plt.subplot(2, 5, i+1)
        plt.imshow(imagenes[i])
        plt.title(f"{class_names[etiquetas[i][0]]}")
        plt.axis('off')
    plt.suptitle("Muestras del Tensor 4D de CIFAR-10", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

# Mostramos las primeras 10 imágenes de entrenamiento
mostrar_imagenes(X_train, y_train, class_names, num_imagenes=10)

# Analicemos una imagen específica en detalle
imagen_idx = 5  # Cambia este índice para explorar diferentes imágenes
imagen_ejemplo = X_train[imagen_idx]
print(f"🔍 Imagen en índice {imagen_idx}:")
print(f"  Clase: {class_names[y_train[imagen_idx][0]]}")
print(f"  Forma del tensor de la imagen: {imagen_ejemplo.shape}")
print(f"  Esto es un tensor 3D: (alto={imagen_ejemplo.shape[0]}, ancho={imagen_ejemplo.shape[1]}, canales={imagen_ejemplo.shape[2]})")

---
## Actividad 2: Extraer un canal de color (tensor 3D) y visualizarlo

Cada imagen es un tensor 3D (alto, ancho, canales). Los canales son:
*   Canal 0: Rojo (Red)
*   Canal 1: Verde (Green)
*   Canal 2: Azul (Blue)

Extraeremos cada canal por separado para entender cómo contribuye a la imagen final.

In [ ]:
# Elegimos una imagen para el análisis
img_idx = 15  # Puedes cambiar este índice
imagen = X_train[img_idx]
etiqueta = class_names[y_train[img_idx][0]]

# Extraemos los canales (tensor 3D -> cada canal es una matriz 2D)
canal_rojo = imagen[:, :, 0]  # Tensor 2D (32, 32)
canal_verde = imagen[:, :, 1] # Tensor 2D (32, 32)
canal_azul = imagen[:, :, 2]  # Tensor 2D (32, 32)

print(f"🔴 Canal Rojo - forma: {canal_rojo.shape}")
print(f"🟢 Canal Verde - forma: {canal_verde.shape}")
print(f"🔵 Canal Azul - forma: {canal_azul.shape}")

# Visualizamos la imagen original y sus canales
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Fila superior: imágenes en color (repetimos la original para contexto)
axes[0, 0].imshow(imagen)
axes[0, 0].set_title(f'Original (color) - {etiqueta}')
axes[0, 0].axis('off')

# Reconstruimos la imagen a partir de un solo canal (poniendo los otros en cero)
imagen_rojo = np.zeros_like(imagen)
imagen_rojo[:, :, 0] = canal_rojo
axes[0, 1].imshow(imagen_rojo.astype(np.uint8))
axes[0, 1].set_title('Solo Canal Rojo')
axes[0, 1].axis('off')

imagen_verde = np.zeros_like(imagen)
imagen_verde[:, :, 1] = canal_verde
axes[0, 2].imshow(imagen_verde.astype(np.uint8))
axes[0, 2].set_title('Solo Canal Verde')
axes[0, 2].axis('off')

# Fila inferior: canales individuales como mapas de calor (escala de grises)
axes[1, 0].imshow(canal_rojo, cmap='Reds')
axes[1, 0].set_title('Mapa de intensidad - Rojo')
axes[1, 0].axis('off')

axes[1, 1].imshow(canal_verde, cmap='Greens')
axes[1, 1].set_title('Mapa de intensidad - Verde')
axes[1, 1].axis('off')

axes[1, 2].imshow(canal_azul, cmap='Blues')
axes[1, 2].set_title('Mapa de intensidad - Azul')
axes[1, 2].axis('off')

plt.suptitle(f'Exploración de Canales de Color (Imagen {img_idx})', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Análisis de canales

Observa cómo cada canal captura diferentes aspectos de la imagen. Por ejemplo:
*   En imágenes de **aviones** o **barcos**, el canal azul puede ser más intenso en el fondo.
*   En **automóviles** rojos, el canal rojo tendrá valores altos en esas áreas.
*   Los **pájaros** y **animales** suelen tener contribuciones variadas en todos los canales.

**Pregunta reflexiva:** ¿Qué canal crees que sería más informativo para distinguir entre un 'gato' y un 'perro'? ¿Depende del color específico de los animales en la imagen?

### Exploración estadística de canales

Calculemos estadísticas básicas de los canales para entender su distribución.

In [ ]:
# Calculamos estadísticas por canal para la imagen seleccionada
print(f"📊 Estadísticas por canal (Imagen {img_idx}):")
for i, (canal, nombre, color) in enumerate([
    (canal_rojo, 'Rojo', 'red'),
    (canal_verde, 'Verde', 'green'),
    (canal_azul, 'Azul', 'blue')
]):
    print(f"  {nombre}:")
    print(f"    Media: {canal.mean():.2f}")
    print(f"    Desviación estándar: {canal.std():.2f}")
    print(f"    Mínimo: {canal.min()}, Máximo: {canal.max()}")

# Histogramas de los canales
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.hist(canal_rojo.flatten(), bins=50, color='red', alpha=0.7, edgecolor='black')
plt.title('Histograma - Canal Rojo')
plt.xlabel('Intensidad')
plt.ylabel('Frecuencia')

plt.subplot(1, 3, 2)
plt.hist(canal_verde.flatten(), bins=50, color='green', alpha=0.7, edgecolor='black')
plt.title('Histograma - Canal Verde')
plt.xlabel('Intensidad')

plt.subplot(1, 3, 3)
plt.hist(canal_azul.flatten(), bins=50, color='blue', alpha=0.7, edgecolor='black')
plt.title('Histograma - Canal Azul')
plt.xlabel('Intensidad')

plt.suptitle(f'Distribución de Intensidades por Canal (Imagen {img_idx})', fontsize=16)
plt.tight_layout()
plt.show()

---
## Actividad 3: Aplanar una imagen y discutir la pérdida de estructura espacial

Aplanar (flatten) una imagen significa convertir el tensor 3D (alto, ancho, canales) en un vector 1D. Esto es necesario para alimentar imágenes a redes neuronales densas (fully-connected), pero tiene un costo: se pierde la información de proximidad espacial.

### 3.1. Aplanamiento de una imagen

In [ ]:
# Tomamos la misma imagen del ejemplo anterior
print(f"Imagen original - forma: {imagen.shape} (tensor 3D: alto x ancho x canales)")

# Aplanamos la imagen a un vector 1D
imagen_aplanada = imagen.flatten()
print(f"Imagen aplanada - forma: {imagen_aplanada.shape} (tensor 1D)")
print(f"Total de píxeles: {len(imagen_aplanada)} (32x32x3 = 3,072 píxeles)")

# Mostramos los primeros valores del vector
print(f"\nPrimeros 20 valores del vector aplanado:")
print(imagen_aplanada[:20])

# Intentemos reconstruir la imagen desde el vector aplanado
imagen_reconstruida = imagen_aplanada.reshape(32, 32, 3)

# Verificamos que la reconstrucción es idéntica
print(f"\n¿Reconstrucción exitosa? {np.array_equal(imagen, imagen_reconstruida)}")

# Visualizamos original vs reconstruido (deberían ser iguales)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(imagen)
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(imagen_reconstruida)
axes[1].set_title('Reconstruida desde vector')
axes[1].axis('off')

plt.suptitle('Aplanamiento y Reconstrucción de Imagen')
plt.tight_layout()
plt.show()

### 3.2. Visualización de la pérdida de estructura

Aunque podemos reconstruir la imagen perfectamente desde el vector aplanado (porque guardamos todos los píxeles), el problema está en cómo un **modelo** procesa esa información.

En el tensor original, los píxeles vecinos están cerca en la estructura 2D/3D. En el vector aplanado, píxeles que estaban muy separados en la imagen original pueden terminar siendo vecinos en el vector, y viceversa. Esto confunde a los modelos densos.

**Demostración:** Comparemos la distancia entre píxeles vecinos en la imagen original vs en el vector aplanado.

In [ ]:
# Seleccionamos un píxel de referencia (por ejemplo, el píxel central)
ref_fila, ref_col = 16, 16  # Centro aproximado

# En la imagen original, sus vecinos inmediatos
vecinos_original = [
    imagen[ref_fila-1, ref_col, 0],  # arriba
    imagen[ref_fila+1, ref_col, 0],  # abajo
    imagen[ref_fila, ref_col-1, 0],  # izquierda
    imagen[ref_fila, ref_col+1, 0]   # derecha
]

# En el vector aplanado, ¿dónde están esos vecinos?
# Calculamos el índice del píxel de referencia en el vector aplanado
indice_ref = ref_fila * 32 * 3 + ref_col * 3  # Considerando canales

# Los vecinos en el vector aplanado (solo mostramos índices)
indice_arriba = (ref_fila-1) * 32 * 3 + ref_col * 3
indice_abajo = (ref_fila+1) * 32 * 3 + ref_col * 3
indice_izq = ref_fila * 32 * 3 + (ref_col-1) * 3
indice_der = ref_fila * 32 * 3 + (ref_col+1) * 3

print("🔍 Comparación de proximidad:")
print(f"  Píxel de referencia en (fila={ref_fila}, col={ref_col}) - índice en vector: {indice_ref}")
print(f"\n  Vecinos en imagen original (distancia espacial = 1 píxel):")
print(f"    Arriba: índice {indice_arriba}, distancia en vector: {abs(indice_ref - indice_arriba)}")
print(f"    Abajo: índice {indice_abajo}, distancia en vector: {abs(indice_ref - indice_abajo)}")
print(f"    Izquierda: índice {indice_izq}, distancia en vector: {abs(indice_ref - indice_izq)}")
print(f"    Derecha: índice {indice_der}, distancia en vector: {abs(indice_ref - indice_der)}")

# Nota: los vecinos verticales (arriba/abajo) están a 96 posiciones de distancia
# porque cada fila tiene 32 columnas * 3 canales = 96 elementos
print(f"\n⚠️ Los vecinos verticales están a 96 posiciones en el vector,")
print(f"   mientras que los horizontales solo a 3 posiciones (cambios de canal).")
print(f"   Esto rompe la simetría espacial que existe en la imagen original.")

### 3.3. Discusión: Implicaciones para el Deep Learning

**¿Por qué es importante la estructura espacial?**

1.  **Localidad:** En imágenes, los píxeles cercanos están fuertemente correlacionados. Un perro tiene orejas cerca de su cabeza, no dispersas por toda la imagen.
2.  **Invariancia a traslaciones:** Un gato sigue siendo un gato sin importar dónde aparezca en la imagen. Las CNN explotan esto compartiendo pesos.
3.  **Jerarquía de características:** Las CNN construyen características de bajo nivel (bordes) a partir de píxeles locales, luego combinan estas en formas, etc.

**¿Qué pasa cuando aplanamos?**

*   Perdemos la noción de "vecindad". Una red fully-connected trataría un píxel en la esquina superior izquierda igual que uno en el centro, sin priorizar relaciones locales.
*   Necesitamos muchos más parámetros para aprender relaciones que las CNN capturan naturalmente con kernels pequeños.
*   La red no es invariante a traslaciones; si movemos el objeto, todos los pesos tendrían que "re-aprender" su posición.

**Ejemplo conceptual:**
Para detectar un ojo, una CNN mira una pequeña región de la imagen (porque sabe que los ojos están cerca de las narices, etc.). Una red densa, al recibir el vector aplanado, tendría que aprender desde cero qué combinaciones de píxeles (sin importar su posición original) forman un ojo, lo cual es mucho más difícil y requiere más datos.

### 3.4. Comparación con MLP (red densa) vs CNN (convolucional)

Aunque aún no hemos visto CNN, visualicemos la diferencia en el número de parámetros.

In [ ]:
# Supongamos que queremos una primera capa con 100 neuronas
num_neuronas = 100

# Para una red densa (MLP) con entrada aplanada
entrada_densa = 32 * 32 * 3  # 3072
parametros_densa = entrada_densa * num_neuronas + num_neuronas  # pesos + sesgos
print(f"🧠 Red Densa (MLP) - Parámetros en primera capa: {parametros_densa:,}")

# Para una capa convolucional típica (kernel 3x3, 3 canales entrada, 100 filtros)
kernel_size = 3
canales_entrada = 3
parametros_conv = kernel_size * kernel_size * canales_entrada * num_neuronas + num_neuronas
print(f"🔷 Capa Convolucional - Parámetros: {parametros_conv:,}")

print(f"\n📊 Comparación:")
print(f"  La red densa tiene {parametros_densa / parametros_conv:.1f} veces más parámetros")
print(f"  para la misma cantidad de neuronas/filtros.")
print(f"  Además, la capa convolucional explota la estructura espacial,")
print(f"  mientras que la densa trata la imagen como un 'saco de píxeles'.")

---
## Ejercicios para el Estudiante

1.  **Exploración de Canales:** Elige una imagen de CIFAR-10 de la clase 'barco' y otra de la clase 'automóvil'. Compara sus histogramas de canales. ¿Qué diferencias observas? ¿En qué canal esperarías ver valores más altos para el 'barco' (piensa en el agua/cielo)?

2.  **Manipulación Creativa de Tensores:** Crea una nueva imagen combinando el canal rojo de una imagen, el canal verde de otra y el canal azul de una tercera. Visualiza el resultado (puedes usar `np.dstack` para combinar canales). ¿La imagen resultante tiene sentido? ¿Por qué sí o por qué no?

3.  **Aplanamiento de un Lote:** Aplana las primeras 10 imágenes de entrenamiento para crear una matriz 2D de forma (10, 3072). ¿Qué representa cada fila? ¿Qué representa cada columna?

4.  **Reflexión Crítica:** Si estuvieras construyendo un modelo para clasificar imágenes de rayos X médicos (donde la estructura espacial es crucial para detectar anomalías), ¿usarías una red densa o una convolucional como primera opción? Justifica tu respuesta basándote en lo aprendido sobre la pérdida de estructura espacial.

---
## Conclusiones de la Sesión

*   Hemos trabajado con **CIFAR-10**, un dataset real de imágenes, confirmando que se organiza como un **tensor 4D** (lote, alto, ancho, canales).
*   Exploramos cómo **extraer y visualizar canales de color** individuales (tensores 3D) y analizamos su contribución a la imagen final.
*   Realizamos operaciones de **slicing y manipulación** sobre los tensores, habilidades fundamentales en el preprocesamiento de datos.
*   Aprendimos a **aplanar imágenes** a vectores 1D y discutimos en profundidad la **pérdida de estructura espacial** que esto conlleva, así como sus implicaciones en el diseño de arquitecturas de deep learning (MLP vs CNN).
*   Esta comprensión sienta las bases para entender por qué las **redes convolucionales** son tan efectivas en tareas de visión por computador: porque respetan y explotan la estructura tensorial de las imágenes.

¡Has dado un paso gigante en la comprensión de cómo la IA "ve" y procesa las imágenes!